# Merge: OpenCity results + MyNeta assets + TCPD party table

**Inputs**
- `clean/opencity_results_clean_2026-10-20.csv`: general election results (notebook 02)
- `interim/opencity_byelections_2014_onwards.csv`: by-election results (notebook 01)
- `interim/candidates_processed.csv`: MyNeta candidates, general and by-election rows
- `raw/TCPD-PPI_1962_2021.csv`: TCPD party table (party id, type, abbreviations)

**Output (`clean/`):** merged general election table, merged by-election table, one combined table, and the MyNeta rows that found no partner.

**Plan:** (1) normalise candidate names, (2) link MyNeta seats to constituency numbers, (3) normalise parties with the TCPD table, (4) parse asset strings, (5) match candidates inside each seat, (6) same for by-elections, (7) save.

In [1]:
from pathlib import Path
import re
import difflib
import numpy as np
import pandas as pd

In [2]:
# folders and input files
project_dir = Path.cwd().parent
raw_dir = project_dir / "raw"
interim_dir = project_dir / "interim"
clean_dir = project_dir / "clean"

COLLECTION_DATE = "2026-10-20"
states = ["Bihar", "Maharashtra", "Uttar Pradesh", "Tamil Nadu"]

results = pd.read_csv(clean_dir / f"opencity_results_clean_{COLLECTION_DATE}.csv")
oc_bye = pd.read_csv(interim_dir / "opencity_byelections_2014_onwards.csv", low_memory=False)
myneta = pd.read_csv(interim_dir / "candidates_processed.csv")
ppi = pd.read_csv(interim_dir / "TCPD-PPI_1962_2021.csv")

# row numbers in the input files (0 = first data row under the header), kept for sanity checks
results["OC_Row"] = results.index
oc_bye["OC_Row"] = oc_bye.index
myneta["MN_Row"] = myneta.index

print("OpenCity general:", results.shape, "| OpenCity by-elections (all states):", oc_bye.shape)
print("MyNeta:", myneta.shape, "| TCPD party table:", ppi.shape)

OpenCity general: (10378, 24) | OpenCity by-elections (all states): (410, 46)
MyNeta: (10395, 16) | TCPD party table: (10153, 21)


### Step 1: candidate names

Both sources write names differently: `Dr.`, `Adv.`, `Thiru.`, `Smt` in front, dots in initials, different case. The same function is used on both sides: upper case, symbols become spaces, titles dropped. Three versions are kept for matching:
- `Name`: cleaned name
- `Name_Key`: the words in alphabetical order (ignores word order)
- `Letters_Key`: all letters sorted, spaces ignored (ignores word order **and** spacing of initials, so `K P MUNUSAMY` equals `KP MUNUSAMY`)

In [3]:
TITLES = {"DR", "PROF", "PROFESSOR", "ADV", "ADVOCATE", "ER", "ENGR", "SHRI", "SRI", "SHREE", "SMT", "SMTI", "SHRIMATI",
          "TMT", "THIRU", "THIRUMATHI", "THIRUVALAR", "SELVI", "MR", "MRS", "MS", "MISS", "COL", "CAPT", "MAJ", "RETD", "RTD"}

def clean_name(name):
    text = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper())
    return " ".join(t for t in text.split() if t not in TITLES)

def title_count(series):
    words = series.astype(str).str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True).str.split()
    return int(words.map(lambda w: any(t in TITLES for t in w)).sum())

myneta["Name"] = myneta["candidate"].map(clean_name)
results["Name"] = results["Candidate"].map(clean_name)

print("names with a title before / after")
print("  MyNeta  :", title_count(myneta["candidate"]), "/", title_count(myneta["Name"]))
print("  OpenCity:", title_count(results["Candidate"]), "/", title_count(results["Name"]))

for frame in (myneta, results):
    frame["Name_Key"] = frame["Name"].map(lambda n: " ".join(sorted(n.split())))
    frame["Letters_Key"] = frame["Name"].map(lambda n: "".join(sorted(n.replace(" ", ""))))

names with a title before / after
  MyNeta  : 433 / 0
  OpenCity: 33 / 0


### Step 2: link MyNeta seats to constituency numbers

MyNeta gives the constituency name (`GAYA (SC)`). OpenCity has the number and a name. The seat name is made comparable (no `(SC)`, no symbols), looked up inside the state, and leftovers are matched by similar spelling. The result is printed.

In [4]:
def seat_norm(s):
    s = re.sub(r"\((SC|ST|GEN)\)", "", str(s), flags=re.I)
    return re.sub(r"[^A-Z0-9]", "", s.upper())

seats = results.drop_duplicates(["State", "Constituency_No"])[["State", "Constituency_No", "Constituency_Name", "Constituency_Type"]]
seat_lookup = {(r.State, seat_norm(r.Constituency_Name)): r.Constituency_No for r in seats.itertuples()}

myneta["State"] = myneta["state"].str.title()
seat_pairs = myneta[["State", "const"]].drop_duplicates()

resolved, fuzzy_seats, unresolved = {}, [], []
for state, const in seat_pairs.itertuples(index=False):
    key = (state, seat_norm(const))
    if key in seat_lookup:
        resolved[(state, const)] = seat_lookup[key]
        continue
    same_state = [k[1] for k in seat_lookup if k[0] == state]
    close = difflib.get_close_matches(key[1], same_state, n=1, cutoff=0.8)
    if close:
        resolved[(state, const)] = seat_lookup[(state, close[0])]
        fuzzy_seats.append((state, const, close[0]))
    else:
        unresolved.append((state, const))

myneta["Constituency_No"] = [resolved.get((s, c)) for s, c in zip(myneta["State"], myneta["const"])]
print("MyNeta seat names:", len(seat_pairs), "| matched exactly:", len(seat_pairs) - len(fuzzy_seats) - len(unresolved))
print("matched by similar spelling:", fuzzy_seats)
print("not matched:", unresolved)

MyNeta seat names: 255 | matched exactly: 250
matched by similar spelling: [('Maharashtra', 'AHMEDNAGAR', 'AHMADNAGAR'), ('Maharashtra', 'HATKANANGALE', 'HATKANANGLE'), ('Tamil Nadu', 'TIRUVALLUR (SC)', 'THIRUVALLUR'), ('Tamil Nadu', 'TIRUVALLUR', 'THIRUVALLUR'), ('Uttar Pradesh', 'GAUTAM BUDHHA NAGAR', 'GAUTAMBUDDHANAGAR')]
not matched: []


In [5]:
# which seats have no general-election MyNeta rows in a year?
general_mn = myneta[~myneta["is_byelection"]]
for year in [2014, 2019, 2024]:
    have = set(general_mn.loc[general_mn["year"] == year, ["State", "Constituency_No"]].dropna().itertuples(index=False, name=None))
    missing = [(s, n, name) for s, n, name in seats[["State", "Constituency_No", "Constituency_Name"]].itertuples(index=False) if (s, n) not in have]
    print(year, "seats without MyNeta rows:", len(missing), missing)

2014 seats without MyNeta rows: 0 []
2019 seats without MyNeta rows: 0 []
2024 seats without MyNeta rows: 0 []


### Step 3: party names with the TCPD party table

Each party string (MyNeta short names, OpenCity codes, 2024 full names) is turned into a comparable key (upper case, symbols removed), and looked up among the TCPD names and abbreviations of Lok Sabha parties. The state is used first, because the same abbreviation can mean different parties in different states.

| Method | Meaning |
|---|---|
| exact | one live TCPD party fits |
| exact-old | only parties that stopped contesting before 2014 fit (kept, but check) |
| alias | a hand-written rule (e.g. `AIADMK` is `ADMK`, split parties go to their parent) |
| ambiguous | more than one live TCPD party fits, the largest is taken and the cases are printed |
| fuzzy | a spelling almost the same (95% similar) was found; printed for review |
| unmapped | not in TCPD (e.g. parties formed after 2021) |

TCPD ends in 2021, so the 2022 Shiv Sena and 2023 NCP splits are mapped to their parent party.

In [6]:
def pnorm(text):
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())

ppi_ls = ppi[ppi["Assembly"] == "Lok_Sabha"].copy()
ppi_ls["State"] = ppi_ls["State_Name"].str.replace("_", " ")

# one row per party id with its main facts
party_info = (ppi_ls.groupby("Party_ID").agg(Party_Name=("Party_Name", "first"), Party_Type=("Party_Type", "first"),
                                              Last_Year=("Last_Year", "max"), Size=("Candidates_Contested", "max")))

# every name and abbreviation points to (party id, state) pairs; full names are tried before abbreviations
name_lookup, abbr_lookup = {}, {}
for r in ppi_ls.itertuples():
    for text in [r.Party_Name]:
        name_lookup.setdefault(pnorm(text), set()).add((r.Party_ID, r.State))
    for text in [r.Frequent_Abbreviation, r.Last_Abbreviation] + str(r.Abbreviations).split("|"):
        key = pnorm(text)
        if key and key != "NAN":
            abbr_lookup.setdefault(key, set()).add((r.Party_ID, r.State))
lookup = {**abbr_lookup, **name_lookup}   # a full name beats an abbreviation

# hand-written aliases (key after pnorm -> key that exists in TCPD)
ALIASES = {"AIADMK": "ADMK", "AAP": "AAAP"}

# full party names (2024 file, MyNeta) that TCPD stores under another name; they point to the TCPD abbreviation
FULL_NAMES = {
    "Bharatiya Janata Party": "BJP", "Indian National Congress": "INC", "Bahujan Samaj Party": "BSP",
    "Samajwadi Party": "SP", "Rashtriya Janata Dal": "RJD", "Janata Dal (United)": "JD(U)",
    "Dravida Munnetra Kazhagam": "DMK", "All India Anna Dravida Munnetra Kazhagam": "ADMK",
    "Nationalist Congress Party": "NCP", "Communist Party of India": "CPI", "Communist Party of India (Marxist)": "CPM",
    "Communist Party of India (Marxist-Leninist) (Liberation)": "CPI(ML)(L)", "Shiv Sena": "SHS",
    "All India Majlis-E-Ittehadul Muslimeen": "AIMIM", "Pattali Makkal Katchi": "PMK",
    "Desiya Murpokku Dravida Kazhagam": "DMDK", "Viduthalai Chiruthaigal Katchi": "VCK",
    "Marumalarchi Dravida Munnetra Kazhagam": "MDMK", "Rashtriya Lok Dal": "RLD", "Naam Tamilar Katchi": "NTK",
    "Indian Union Muslim League": "IUML", "Apna Dal (Soneylal)": "ADAL", "Hindustani Awam Morcha (Secular)": "HAMS",
    "Lok Janshakti Party": "LJP", "Independent": "IND", "None of the Above": "NOTA",
}
ALIASES.update({pnorm(name): pnorm(code) for name, code in FULL_NAMES.items()})
FACTION_RULES = [("UDDHAV", "SHS"), ("SHARADCHANDRA", "NCP"), ("NCPSP", "NCP"), ("RAMVILAS", "LJP"), ("LJPRV", "LJP")]

def resolve_party(raw, state):
    key = pnorm(raw)
    method = "exact"
    for fragment, parent in FACTION_RULES:
        if fragment in key:
            key, method = parent, "alias"
    if key in ALIASES:
        key, method = ALIASES[key], "alias"

    options = lookup.get(key)
    if not options:
        close = difflib.get_close_matches(key, list(lookup), n=1, cutoff=0.95)
        if not close:
            return np.nan, "unmapped", []
        options, method = lookup[close[0]], "fuzzy"

    ids = {pid for pid, st in options if st in (state, "All States")} or {pid for pid, _ in options}
    live = [p for p in ids if party_info.loc[p, "Last_Year"] >= 2014]
    if len(live) == 1:
        return live[0], method, []
    if len(live) > 1:
        ranked = sorted(live, key=lambda p: party_info.loc[p, "Size"], reverse=True)
        return ranked[0], "ambiguous", ranked
    best = max(ids, key=lambda p: party_info.loc[p, "Size"])
    return best, ("exact-old" if method == "exact" else method), []

In [7]:
# the party strings of all three sources
results["Party_Raw"] = results["Party_Full"]
myneta["Party_Raw"] = myneta["party"]
# the by-election file already has its own Party_ID column; keep it under another name so ours is the only Party_ID
oc_bye = oc_bye.rename(columns={"Party_ID": "Party_ID_in_file"})
oc_bye["State"] = oc_bye["State_Name"].str.replace("_", " ")
oc_bye["Party_Raw"] = oc_bye["Party"]

pairs = pd.concat([results[["State", "Party_Raw"]], myneta[["State", "Party_Raw"]], oc_bye[["State", "Party_Raw"]]]).drop_duplicates()

rows = []
for state, raw in pairs.itertuples(index=False):
    pid, method, options = resolve_party(raw, state)
    rows.append({"State": state, "Party_Raw": raw, "Party_ID": pid, "Party_Method": method, "Options": options})
party_map = pd.DataFrame(rows)
party_map["Party_Canonical"] = party_map["Party_ID"].map(party_info["Party_Name"])
party_map["Party_Type"] = party_map["Party_ID"].map(party_info["Party_Type"]).fillna("Not in TCPD")

print("distinct (state, party string) pairs:", len(party_map))
print(party_map["Party_Method"].value_counts().to_string())

distinct (state, party string) pairs: 1742
Party_Method
exact        1333
unmapped      277
alias          54
ambiguous      38
fuzzy          32
exact-old       8


In [8]:
# the cases to look at: ambiguous, fuzzy, exact-old and unmapped
for method in ["ambiguous", "fuzzy", "exact-old"]:
    part = party_map[party_map["Party_Method"] == method].drop_duplicates("Party_Raw")
    print(f"\n{method}: {len(part)} strings")
    for r in part.head(25).itertuples():
        names = [party_info.loc[p, "Party_Name"] for p in r.Options] if r.Options else []
        print(f"  {r.Party_Raw!r:55} -> {r.Party_Canonical!r} ({r.Party_Type}) {('| other options: ' + str(names)) if names else ''}")

rows_per_string = pd.concat([results["Party_Raw"], myneta["Party_Raw"], oc_bye["Party_Raw"]]).value_counts()
oc_unmapped = party_map[(party_map["Party_Method"] == "unmapped") & party_map["Party_Raw"].isin(results["Party_Raw"])]
print("unmapped strings that come from the OpenCity results:", oc_unmapped["Party_Raw"].tolist()[:25])
unmapped = party_map[party_map["Party_Method"] == "unmapped"].drop_duplicates("Party_Raw")
print(f"\nunmapped: {len(unmapped)} strings, covering {int(rows_per_string.reindex(unmapped['Party_Raw']).sum())} rows")
print(rows_per_string.reindex(unmapped["Party_Raw"]).sort_values(ascending=False).head(20).to_string())


ambiguous: 19 strings
  'JD(U)'                                                 -> 'Janata Dal  (United)' (State-based Party) | other options: ['Janata Dal  (United)', 'Janata Party (United)']
  'CPI(ML)(L)'                                            -> 'Communist Party of India  (Marxist-Leninist)  (Liberation)' (Local Party) | other options: ['Communist Party of India  (Marxist-Leninist)  (Liberation)', 'Communist Party of India  (Marxist-Leninist)']
  'VIP'                                                   -> 'Vanchitsamaj Insaaf Party' (Local Party) | other options: ['Vanchitsamaj Insaaf Party', 'Vikassheel Insaan Party']
  'CPIM'                                                  -> 'COMMUNIST MARXIST PARTY OF INDIA' (State-based Party) | other options: ['COMMUNIST MARXIST PARTY OF INDIA', 'Communist Party of India (Marxist-Leninist) Red star']
  'ASP'                                                   -> 'Ambedkar Samaj Party' (Local Party) | other options: ['Ambedkar Samaj Party',

In [9]:
# attach the party facts to every source
def add_party(frame, state_col="State"):
    merged = frame.merge(party_map.drop(columns="Options"), on=[state_col, "Party_Raw"], how="left")
    assert len(merged) == len(frame)
    return merged

results = add_party(results)
myneta = add_party(myneta)
oc_bye = add_party(oc_bye)

print(results.groupby("Year")["Party_Type"].value_counts().unstack().to_string())

Party_Type  Independents  Local Party  National Party  Not in TCPD  State-based Party
Year                                                                                 
2014              1503.0        893.0           307.0          NaN              934.0
2019              1474.0        426.0           244.0        731.0              447.0
2024              1698.0        230.0           208.0        799.0              484.0


### Step 4: asset strings to numbers

`Rs 1,60,59,806 ~ 1 Crore+` becomes 16059806 (rupees). `Nil` becomes 0 and is flagged. Anything else that cannot be read becomes empty and is counted.

In [10]:
def parse_rs(text):
    t = str(text).replace("\xa0", " ").strip()
    if t.lower() == "nil":
        return 0.0
    found = re.search(r"Rs\s*([\d,]+)", t)
    return float(found.group(1).replace(",", "")) if found else np.nan

for col, new in [("total_assets", "Assets_Rs"), ("liabilities", "Liabilities_Rs")]:
    myneta[new] = myneta[col].map(parse_rs)
    print(new, "| could not be read:", myneta[new].isna().sum(), "| zero or nil:", (myneta[new] == 0).sum())

myneta["Assets_Nil"] = myneta["total_assets"].astype(str).str.strip().str.lower() == "nil"
print(myneta[["Assets_Rs", "Liabilities_Rs"]].describe().round(0).to_string())

Assets_Rs | could not be read: 0 | zero or nil: 86
Liabilities_Rs | could not be read: 0 | zero or nil: 5772
          Assets_Rs  Liabilities_Rs
count  1.039500e+04    1.039500e+04
mean   3.925396e+07    5.436462e+06
std    2.217224e+08    7.390159e+07
min    0.000000e+00    0.000000e+00
25%    5.100000e+05    0.000000e+00
50%    2.880137e+06    0.000000e+00
75%    1.425888e+07    6.000000e+05
max    1.107583e+10    6.495000e+09


### Step 5: match candidates inside each seat

For each (election, seat), the OpenCity candidates are paired one-to-one with the MyNeta candidates. Rules, in order (a row is used only once):

1. **Same name key** (same party first when two candidates share a name).

Rules 1 to 3 are not applied to two candidates whose TCPD parties are both known and different. The one exception: with an identical name (rules 1 and 2), a clash between an independent and a registered party is allowed, because the two files sometimes label a small-party candidate as independent. Other clashes stay unmatched and are printed.
2. **Same letters key** (same letters in any order, spacing of initials ignored).
3. **Initials and full name**: one side is initials plus surname (`A S PATIL`), the other is the full name (`ASHOKRAO SHANKARRAO PATIL`); surnames must be (almost) equal and the initials must be the first letters of the other names.
4. **Similar name key**, `difflib` similarity at least 0.85.
5. **Same party (not independent) and similarity at least 0.70.**
6. **Last one left on both sides with the same party (not independent).**

Rows that no rule pairs are printed, with the most plausible partner and the TCPD party of both.

In [11]:
IND_ID = party_info.index[party_info["Party_Type"] == "Independents"][0]

def similarity(a, b):
    return difflib.SequenceMatcher(None, a, b).ratio()

def same_party(o, m):
    return pd.notna(o) and o == m and o != IND_ID

BIG_TYPES = {"National Party", "State-based Party"}

def party_conflict(o, m):
    # both parties known and different, and at least one is a national or state party.
    # Labels of small parties differ between the sources, so a clash among small parties does not count.
    if pd.isna(o) or pd.isna(m) or o == m:
        return False
    return bool({party_info.at[o, "Party_Type"], party_info.at[m, "Party_Type"]} & BIG_TYPES)

def initials_match(short, long):
    # `short` looks like initials + surname (A S PATIL), `long` is the full name (ASHOKRAO SHANKARRAO PATIL)
    ts, tl = short.split(), long.split()
    full = [t for t in ts if len(t) > 1]
    initials = [t for t in ts if len(t) == 1]
    if not full or not initials or len(tl) < len(ts):
        return False
    rest = list(tl)
    for t in full:
        best = max(rest, key=lambda w: similarity(t, w))
        if similarity(t, best) < 0.85:
            return False
        rest.remove(best)
    firsts = [w[0] for w in rest]
    for i in initials:
        if i not in firsts:
            return False
        firsts.remove(i)
    return True

def match_seat(oc, mn):
    # oc and mn: the rows of one seat; returns (oc_index, mn_index, rule, score) tuples
    pairs, used_oc, used_mn = [], set(), set()

    def take(o, m, rule, score):
        pairs.append((o, m, rule, score))
        used_oc.add(o)
        used_mn.add(m)

    for rule, col in [(1, "Name_Key"), (2, "Letters_Key")]:
        for m in mn.index:
            if m in used_mn:
                continue
            found = [o for o in oc.index if o not in used_oc and oc.at[o, col] == mn.at[m, col]
                     and not party_conflict(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"])]
            if found:
                preferred = [o for o in found if oc.at[o, "Party_ID"] == mn.at[m, "Party_ID"]]
                take((preferred or found)[0], m, rule, 1.0)

    scored = []
    for m in mn.index:
        for o in oc.index:
            if m in used_mn or o in used_oc:
                continue
            ratio = similarity(oc.at[o, "Name_Key"], mn.at[m, "Name_Key"])
            party_ok = same_party(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"])
            conflict = party_conflict(oc.at[o, "Party_ID"], mn.at[m, "Party_ID"])
            names = (oc.at[o, "Name"], mn.at[m, "Name"])
            if conflict:
                continue
            if initials_match(names[0], names[1]) or initials_match(names[1], names[0]):
                scored.append((0.95 + 0.05 * party_ok, o, m, 3, max(ratio, 0.95)))
            elif ratio >= 0.85:
                scored.append((ratio + 0.05 * party_ok, o, m, 4, ratio))
            elif party_ok and ratio >= 0.70:
                scored.append((ratio, o, m, 5, ratio))
    for _, o, m, rule, ratio in sorted(scored, reverse=True):
        if o not in used_oc and m not in used_mn:
            take(o, m, rule, ratio)

    left_oc = [o for o in oc.index if o not in used_oc]
    left_mn = [m for m in mn.index if m not in used_mn]
    if len(left_oc) == 1 and len(left_mn) == 1 and same_party(oc.at[left_oc[0], "Party_ID"], mn.at[left_mn[0], "Party_ID"]):
        take(left_oc[0], left_mn[0], 6, similarity(oc.at[left_oc[0], "Name_Key"], mn.at[left_mn[0], "Name_Key"]))
    return pairs

def run_matching(oc_frame, mn_frame, keys):
    oc_groups = {k: g for k, g in oc_frame.groupby(keys)}
    all_pairs, mn_unmatched, oc_unmatched = [], [], []
    for k, mn_g in mn_frame.groupby(keys):
        if k not in oc_groups:
            mn_unmatched.extend(mn_g.index)
            continue
        oc_g = oc_groups[k]
        pairs = match_seat(oc_g, mn_g)
        all_pairs.extend(pairs)
        mn_unmatched.extend(set(mn_g.index) - {p[1] for p in pairs})
        oc_unmatched.extend(set(oc_g.index) - {p[0] for p in pairs})
    return all_pairs, mn_unmatched, oc_unmatched

In [12]:
# general election: OpenCity rows against MyNeta general rows
mn_general = myneta[~myneta["is_byelection"] & myneta["Constituency_No"].notna()].copy()
mn_general["Constituency_No"] = mn_general["Constituency_No"].astype(int)

keys = ["Year", "State", "Constituency_No"]
mn_general = mn_general.rename(columns={"year": "Year"})
g_pairs, g_mn_left, g_oc_left = run_matching(results, mn_general, keys)

print("OpenCity candidate rows:", len(results), "| MyNeta general rows:", len(mn_general))
print("matched pairs:", len(g_pairs), "| MyNeta rows left:", len(g_mn_left), "| OpenCity rows left:", len(g_oc_left))
print("pairs by rule:", pd.Series([p[2] for p in g_pairs]).value_counts().sort_index().to_dict())
print("matched share of OpenCity rows by year:")
print((results.loc[[p[0] for p in g_pairs], "Year"].value_counts() / results["Year"].value_counts()).round(3).to_dict())

OpenCity candidate rows: 10378 | MyNeta general rows: 10228
matched pairs: 9957 | MyNeta rows left: 271 | OpenCity rows left: 421
pairs by rule: {1: 8963, 2: 151, 3: 82, 4: 517, 5: 195, 6: 49}
matched share of OpenCity rows by year:
{2014: 0.954, 2024: 0.986, 2019: 0.938}


In [13]:
# two checks on the matched pairs: does the winner agree, does the party agree?
check = pd.DataFrame(g_pairs, columns=["oc", "mn", "rule", "score"])
check["oc_winner"] = results.loc[check["oc"], "Winner"].values
check["mn_winner"] = mn_general.loc[check["mn"], "is_winner"].values
check["oc_party"] = results.loc[check["oc"], "Party_ID"].values
check["mn_party"] = mn_general.loc[check["mn"], "Party_ID"].values

print("winner flag agrees:", round((check["oc_winner"] == check["mn_winner"]).mean(), 4))
both = check[check["oc_party"].notna() & check["mn_party"].notna()]
print("TCPD party id agrees (differences among small parties are allowed):", round((both["oc_party"] == both["mn_party"]).mean(), 4), "of", len(both), "pairs with both known")
print("\nby rule:")
print(check.groupby("rule").agg(pairs=("oc", "size"), winner_agree=("oc_winner", lambda s: 0), party_agree=("oc_party", "size")).drop(columns=["winner_agree", "party_agree"]).to_string())

bad = check[check["oc_winner"] != check["mn_winner"]]
print("\npairs where the winner flag disagrees:", len(bad))
cols = ["Year", "Constituency_Name", "Name", "Party_Raw"]
for r in bad.head(12).itertuples():
    print(" ", results.loc[r.oc, cols].tolist(), "<->", mn_general.loc[r.mn, ["Name", "Party_Raw", "is_winner"]].tolist(), "rule", r.rule, round(r.score, 2))

winner flag agrees: 1.0
TCPD party id agrees (differences among small parties are allowed): 0.9975 of 9417 pairs with both known

by rule:
      pairs
rule       
1      8963
2       151
3        82
4       517
5       195
6        49

pairs where the winner flag disagrees: 0


In [14]:
# ambiguous cases: rows left over, with the most plausible partner in the same seat
def show_leftovers(oc_frame, mn_frame, mn_left, oc_left, keys, limit=40):
    oc_groups = {k: g.loc[g.index.isin(oc_left)] for k, g in oc_frame.groupby(keys)}
    rows = []
    for m in mn_left:
        row = mn_frame.loc[m]
        k = tuple(row[c] for c in keys)
        candidates = oc_groups.get(k)
        best, ratio = None, 0.0
        if candidates is not None and len(candidates):
            scores = [(similarity(row["Name_Key"], c), i) for i, c in candidates["Name_Key"].items()]
            ratio, best = max(scores)
        rows.append({
            "Year": row[keys[0]], "Seat": row["const"], "MyNeta name": row["Name"], "MyNeta party": row["Party_Raw"],
            "TCPD party": row["Party_Canonical"], "MyNeta winner": row["is_winner"],
            "Best OpenCity name": oc_frame.at[best, "Name"] if best is not None else None,
            "OC party": oc_frame.at[best, "Party_Raw"] if best is not None else None,
            "OC TCPD party": oc_frame.at[best, "Party_Canonical"] if best is not None else None,
            "similarity": round(ratio, 2)})
    table = pd.DataFrame(rows)
    if len(table):
        table = table.sort_values("similarity", ascending=False)
    return table

left_table = show_leftovers(results, mn_general, g_mn_left, g_oc_left, keys)
print("MyNeta general rows without a partner:", len(left_table))
print(left_table.head(40).to_string(index=False))

MyNeta general rows without a partner: 271
 Year                 Seat                                  MyNeta name                 MyNeta party                    TCPD party  MyNeta winner                     Best OpenCity name           OC party                 OC TCPD party  similarity
 2019               BALLIA                MAJOR RAMESH CHANDRA UPADHYAY Akhil Bharat Hindu Mahasabha AKHIL BHARAT HINDU MAHA SABHA          False          MAJOR RAMESH CHANDRA UPADHYAY                IND                   Independent        1.00
 2019            MADHUBANI                                SHAKEEL AHMAD                          INC                      CONGRESS          False                          SHAKEEL AHMAD                IND                   Independent        1.00
 2014             VARANASI                             RAM LAKHAN GUPTA              Jantantra Party               Jantantra Party          False                       RAM LAKHAN GUPTA                JAP                

### Step 6: build the merged general election table

OpenCity rows are the base. Matched rows get the MyNeta columns (prefix `MN_`), unmatched rows keep empty MyNeta columns. MyNeta rows without a partner are saved separately.

In [15]:
mn_cols = {"candidate": "MN_Candidate", "party": "MN_Party", "education": "MN_Education", "age": "MN_Age",
           "criminal_cases": "MN_Criminal_Cases", "Assets_Rs": "Assets_Rs", "Liabilities_Rs": "Liabilities_Rs",
           "Assets_Nil": "Assets_Nil", "is_winner": "MN_Is_Winner", "MN_Row": "MN_Row"}

def attach(oc_frame, mn_frame, pairs):
    link = pd.DataFrame(pairs, columns=["oc", "mn", "Match_Rule", "Match_Score"]).set_index("oc")
    out = oc_frame.join(link[["Match_Rule", "Match_Score"]])
    mn_part = mn_frame.loc[link["mn"], list(mn_cols)].rename(columns=mn_cols)
    mn_part.index = link.index
    out = out.join(mn_part)
    out["Has_Assets"] = out["Assets_Rs"].notna()
    return out

general = attach(results, mn_general, g_pairs)
general = general.rename(columns={"Year": "Election_Year"})
general["Term"] = general["Election_Year"]
general["Is_ByElection"] = False
general["OC_File"] = "opencity_results_clean"
general["Bye_Year"] = np.nan
print("general election rows:", len(general), "| with assets:", int(general["Has_Assets"].sum()))
print(general.groupby("Election_Year")["Has_Assets"].agg(["size", "sum", "mean"]).round(3).to_string())

general election rows: 10378 | with assets: 9957
               size   sum   mean
Election_Year                   
2014           3637  3470  0.954
2019           3322  3115  0.938
2024           3419  3372  0.986


### Step 7: by-elections

The OpenCity by-election file holds Lok Sabha by-elections up to 2021, so the by-elections of 2022 (Azamgarh, Rampur, Mainpuri) and 2024 (Nanded) in MyNeta have no OpenCity partner and are reported. A by-election is identified by state + seat + the year it was held (from the MyNeta date). NOTA is removed and position and margin are recomputed, as for the general election.

In [16]:
bye = oc_bye[oc_bye["State"].isin(states) & (oc_bye["Poll_No"] > 0)].copy()
bye["Is_NOTA"] = bye["Party"].eq("NOTA") | bye["Candidate"].str.upper().isin(["NOTA", "NONE OF THE ABOVE"])
print("OpenCity by-election rows in the four states:", len(bye), "| NOTA rows removed:", int(bye["Is_NOTA"].sum()))
bye = bye[~bye["Is_NOTA"]].copy()

bye["Name"] = bye["Candidate"].map(clean_name)
bye["Name_Key"] = bye["Name"].map(lambda n: " ".join(sorted(n.split())))
bye["Letters_Key"] = bye["Name"].map(lambda n: "".join(sorted(n.replace(" ", ""))))

bye = bye.rename(columns={"Year": "Bye_Year", "Valid_Votes": "Seat_Valid_Votes"})
bye["Constituency_Name"] = [dict(zip(zip(seats["State"], seats["Constituency_No"]), seats["Constituency_Name"])).get(k, n)
                            for k, n in zip(zip(bye["State"], bye["Constituency_No"]), bye["Constituency_Name"])]

seat_cols = ["State", "Constituency_No", "Bye_Year"]
bye["Position"] = bye.groupby(seat_cols)["Votes"].rank(method="min", ascending=False).astype(int)
bye["Winner"] = bye["Position"] == 1
first = bye[bye["Position"] == 1].set_index(seat_cols)["Votes"]
second = bye[bye["Position"] == 2].set_index(seat_cols)["Votes"]
bye = bye.drop(columns="Margin").merge((first - second).rename("Margin"), left_on=seat_cols, right_index=True, how="left")
bye["Votes"] = bye["Votes"].astype(int)
bye["Incumbent"] = bye["Incumbent"].map(lambda v: np.nan if pd.isna(v) else str(v) == "True")

print("by-election seats:", bye.groupby(seat_cols).ngroups, "| candidate rows:", len(bye))
print(bye.groupby(seat_cols)["Winner"].sum().eq(1).all(), "<- exactly one winner per by-election")

mn_bye = myneta[myneta["is_byelection"] & myneta["Constituency_No"].notna()].copy()
mn_bye["Constituency_No"] = mn_bye["Constituency_No"].astype(int)
mn_bye["Bye_Year"] = mn_bye["bye_election_date"].astype(str).str[-4:].astype(int)
print("MyNeta by-election rows:", len(mn_bye), "| seats:", mn_bye.groupby(seat_cols).ngroups)

OpenCity by-election rows in the four states: 143 | NOTA rows removed: 9
by-election seats: 12 | candidate rows: 134
True <- exactly one winner per by-election
MyNeta by-election rows: 167 | seats: 16


In [17]:
b_pairs, b_mn_left, b_oc_left = run_matching(bye, mn_bye, seat_cols)
print("matched pairs:", len(b_pairs), "| MyNeta rows left:", len(b_mn_left), "| OpenCity rows left:", len(b_oc_left))

# which by-elections exist on one side only?
oc_events = set(bye.groupby(seat_cols).groups)
mn_events = set(mn_bye.groupby(seat_cols).groups)
print("only in MyNeta (no OpenCity data):", sorted(mn_events - oc_events))
print("only in OpenCity (no MyNeta rows):", sorted(oc_events - mn_events))

check = pd.DataFrame(b_pairs, columns=["oc", "mn", "rule", "score"])
print("winner flag agrees:", (bye.loc[check["oc"], "Winner"].values == mn_bye.loc[check["mn"], "is_winner"].values).mean())
print("\nby-election rows still without a partner (inside seats found on both sides):")
inside = [m for m in b_mn_left if (mn_bye.at[m, "State"], int(mn_bye.at[m, "Constituency_No"]), int(mn_bye.at[m, "Bye_Year"])) in oc_events]
print(show_leftovers(bye, mn_bye, inside, b_oc_left, seat_cols).to_string(index=False) if inside else "none")

matched pairs: 118 | MyNeta rows left: 49 | OpenCity rows left: 16
only in MyNeta (no OpenCity data): [('Maharashtra', 16, 2024), ('Uttar Pradesh', 7, 2022), ('Uttar Pradesh', 21, 2022), ('Uttar Pradesh', 69, 2022)]
only in OpenCity (no MyNeta rows): []
winner flag agrees: 1.0

by-election rows still without a partner (inside seats found on both sides):
         Year     Seat        MyNeta name                   MyNeta party             TCPD party  MyNeta winner             Best OpenCity name OC party         OC TCPD party  similarity
Uttar Pradesh  KAIRANA      PRITI KASHYAP      Jay Hind Jay Bharat Party                    NaN          False           KUMARI PRITI KASHYAP    JHJBP                   NaN        0.79
Uttar Pradesh  PHULPUR    RAIS AHMAD KHAN         Parivartan Samaj Party Parivartan Samaj Party          False       PATRAKAR RAIS AHMAD KHAN      PSP Praja Socialist Party        0.77
Uttar Pradesh  PHULPUR    VARUN KUMAR PAL                            PSP  Praja Socialist

In [18]:
byelection = attach(bye, mn_bye, b_pairs)
byelection["Election_Year"] = byelection["Bye_Year"]
byelection["Term"] = np.where(byelection["Bye_Year"] <= 2018, 2014, 2019)
byelection["Is_ByElection"] = True
byelection["OC_File"] = "opencity_byelections_2014_onwards"
byelection["Party_Full"] = byelection["Party_Raw"]
print("by-election rows:", len(byelection), "| with assets:", int(byelection["Has_Assets"].sum()))
print(byelection.groupby(["Bye_Year", "Constituency_Name"])[["Winner", "Has_Assets"]].agg({"Winner": "sum", "Has_Assets": "mean"}).round(2).to_string())

by-election rows: 134 | with assets: 118
                             Winner  Has_Assets
Bye_Year Constituency_Name                     
2014     BEED                     1        0.92
         MAINPURI                 1        0.92
2018     ARARIA                   1        1.00
         BHANDARA - GONDIYA       1        1.00
         GORAKHPUR                1        1.00
         KAIRANA                  1        0.92
         PALGHAR                  1        1.00
         PHULPUR                  1        0.82
2019     SAMASTIPUR               1        0.88
         SATARA                   1        1.00
2020     VALMIKI NAGAR            1        1.00
2021     KANNIYAKUMARI            1        0.33


### Step 8: combine, check and save

`OC_Row` and `MN_Row` are the row numbers (0 = first data row) in the OpenCity file named in `OC_File` and in `candidates_processed.csv`. `MN_Row` is empty for rows without a MyNeta partner.

In [19]:
columns = ["OC_File", "OC_Row", "MN_Row", "State", "Constituency_No", "Constituency_Name", "Constituency_Type", "Term", "Election_Year", "Is_ByElection", "Bye_Year",
           "Person_ID", "Candidate", "Candidate_Key", "Namesake_In_Seat", "Sex", "Incumbent",
           "Party", "Party_Full", "Party_ID", "Party_Canonical", "Party_Type", "Party_Method", "Party_Group",
           "Position", "Winner", "Votes", "Vote_Share_Percentage", "Margin", "Seat_Valid_Votes",
           "MN_Candidate", "MN_Party", "MN_Education", "MN_Age", "MN_Criminal_Cases", "Assets_Rs", "Liabilities_Rs", "Assets_Nil",
           "Has_Assets", "Match_Rule", "Match_Score"]

general_out = general.reindex(columns=columns)
bye_out = byelection.reindex(columns=columns)
all_out = pd.concat([general_out, bye_out], ignore_index=True)

# MyNeta rows that found no partner (general and by-election)
unmatched = pd.concat([mn_general.loc[g_mn_left].assign(Reason="no partner in seat"),
                       mn_bye.loc[b_mn_left].assign(Reason="no partner in seat or by-election not in OpenCity")])
unmatched = unmatched[["MN_Row", "year", "State", "const", "Constituency_No", "candidate", "party", "Party_Canonical", "is_winner", "is_byelection", "bye_election_date", "Assets_Rs", "Reason"]]
mn_no_seat = myneta[myneta["Constituency_No"].isna()]
print("MyNeta rows without any seat number:", len(mn_no_seat))

general_out.to_csv(clean_dir / f"merged_general_{COLLECTION_DATE}.csv", index=False)
bye_out.to_csv(clean_dir / f"merged_byelection_{COLLECTION_DATE}.csv", index=False)
all_out.to_csv(clean_dir / f"merged_all_{COLLECTION_DATE}.csv", index=False)
unmatched.to_csv(clean_dir / f"merged_unmatched_myneta_{COLLECTION_DATE}.csv", index=False)
party_map.drop(columns="Options").to_csv(clean_dir / f"party_map_{COLLECTION_DATE}.csv", index=False)

print("saved:", len(general_out), "general rows,", len(bye_out), "by-election rows,", len(unmatched), "unmatched MyNeta rows")

MyNeta rows without any seat number: 0
saved: 10378 general rows, 134 by-election rows, 320 unmatched MyNeta rows


In [20]:
# quick look: assets of winners and others, general elections (rupees crore, median)
view = general_out[general_out["Has_Assets"]].copy()
view["Assets_Crore"] = view["Assets_Rs"] / 1e7
print(view.groupby(["Election_Year", "Winner"])["Assets_Crore"].agg(["size", "median"]).round(2).unstack().to_string())
print("\nrows per party type:")
print(all_out.groupby("Is_ByElection")["Party_Type"].value_counts(dropna=False).unstack(0).to_string())

               size       median      
Winner        False True   False True 
Election_Year                         
2014           3271   199   0.21  3.36
2019           2912   203   0.23  6.58
2024           3165   207   0.27  9.27

rows per party type:
Is_ByElection      False  True 
Party_Type                     
Independents        4675     65
Local Party         1549     17
National Party       759     19
Not in TCPD         1530     21
State-based Party   1865     12
